# Unidad 5 · Colab 2 de 3
## Automatización de navegación web en sitios dinámicos: Playwright y Selenium

**Objetivos de este notebook**

- Entender qué cambia entre un sitio estático y uno dinámico (renderizado con JavaScript).
- Configurar y usar **Playwright** para automatizar un navegador real.
- Ubicar elementos con locators, interactuar (click, completar formularios) y esperar contenido dinámico.
- Conocer **Selenium** como alternativa y sus diferencias con Playwright.
- Correr el navegador en modo headless.

> **Nivel:** intermedio. Instalar navegadores en Colab lleva un par de minutos — es normal.

---

## 1. Sitios estáticos vs. dinámicos

En el Colab 1 vimos que `requests` solo trae el HTML inicial que devuelve el servidor. Muchos sitios modernos (React, Vue, Angular) arman gran parte del contenido **en el navegador**, ejecutando JavaScript después de esa respuesta inicial — si hacés `requests.get()` a esos sitios, el HTML que recibís puede estar casi vacío.

Para esos casos necesitás un **navegador real (o headless)** que ejecute el JavaScript, tal como lo haría un usuario. Ejemplo comparable: [quotes.toscrape.com](http://quotes.toscrape.com) (estático) vs [quotes.toscrape.com/js/](http://quotes.toscrape.com/js/) (las mismas citas, pero renderizadas con JavaScript).

## 2. Playwright vs. Selenium

| | Playwright | Selenium |
|---|---|---|
| Origen | Microsoft (2020) | Proyecto histórico (2004), estándar W3C WebDriver |
| Velocidad / estabilidad | Auto-waiting nativo, generalmente más rápido | Requiere esperas explícitas más seguido |
| Navegadores | Chromium, Firefox, WebKit con un solo API | Chrome, Firefox, Edge, Safari (vía drivers) |
| Instalación | `pip install playwright` + `playwright install` | `pip install selenium` + gestor de driver |
| Cuándo usarlo | Proyectos nuevos, scraping moderno | Legacy, ecosistema más grande de integraciones |

Documentación oficial: [Playwright para Python](https://playwright.dev/python/docs/intro) · [Selenium](https://www.selenium.dev/documentation/)

En este notebook usamos Playwright como herramienta principal y mostramos el equivalente en Selenium para que puedas comparar.

## 3. Primeros pasos con Playwright

```bash
pip install playwright
playwright install chromium --with-deps
```

En Colab, instalá con `!` antes de cada comando. La primera vez tarda porque descarga el navegador.

In [ ]:
!pip install -q playwright
!playwright install chromium --with-deps

In [ ]:
import asyncio
from playwright.async_api import async_playwright

async def ejemplo():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('http://quotes.toscrape.com/js/')
        await page.wait_for_selector('.quote')
        titulo = await page.title()
        cantidad = await page.locator('.quote').count()
        print(titulo, '- citas encontradas:', cantidad)
        await browser.close()

await ejemplo()

## 4. Locators en Playwright

`page.locator(selector)` acepta CSS o XPath (con prefijo `xpath=`). A diferencia de `requests` + `BeautifulSoup`, los locators son *perezosos*: no buscan el elemento hasta que hacés una acción, y esperan automáticamente a que exista y sea interactuable.

```python
await page.locator('.quote .text').first.text_content()
```

Para XPath sería algo como: `xpath=//span[contains(@class, 'text')]`.

Documentación oficial: [Locators](https://playwright.dev/python/docs/locators)

### Ejercicio 1 — Extraer citas con Playwright

Usando `http://quotes.toscrape.com/js/`, completá una función que devuelva una lista de diccionarios `{'cita': ..., 'autor': ...}` para todas las citas de la primera página, usando locators de Playwright (no BeautifulSoup).

In [ ]:
async def extraer_citas():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('http://quotes.toscrape.com/js/')
        await page.wait_for_selector('.quote')
        # TODO: recorre page.locator('.quote') y extrae texto y autor de cada una
        await browser.close()
        return []

await extraer_citas()

<details>
<summary>💡 Ver solución</summary>

```python
async def extraer_citas():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('http://quotes.toscrape.com/js/')
        await page.wait_for_selector('.quote')

        citas = []
        bloques = page.locator('.quote')
        n = await bloques.count()
        for i in range(n):
            bloque = bloques.nth(i)
            texto = await bloque.locator('.text').text_content()
            autor = await bloque.locator('.author').text_content()
            citas.append({'cita': texto, 'autor': autor})

        await browser.close()
        return citas

await extraer_citas()
```

</details>

## 5. Interacciones: click, formularios, esperas

```python
await page.fill('#username', 'admin')
await page.fill('#password', 'admin')
await page.click('input[type=submit]')
await page.wait_for_load_state('networkidle')
```

El sitio [quotes.toscrape.com/login](http://quotes.toscrape.com/login) acepta cualquier usuario/contraseña, ideal para practicar.

### Ejercicio 2 — Automatizar un login

Completá el script para: ir a `http://quotes.toscrape.com/login`, completar usuario y contraseña con cualquier valor, hacer click en el botón, y confirmar que apareció el link *Logout* (señal de que el login funcionó).

In [ ]:
async def login():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('http://quotes.toscrape.com/login')
        # TODO: completa el formulario y hace click en submit
        # TODO: verifica que exista el link Logout
        await browser.close()

await login()

<details>
<summary>💡 Ver solución</summary>

```python
async def login():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('http://quotes.toscrape.com/login')
        await page.fill('#username', 'alumno')
        await page.fill('#password', 'curso123')
        await page.click('input[type=submit]')
        logout_visible = await page.locator('text=Logout').count()
        print('Login exitoso' if logout_visible > 0 else 'Login fallo')
        await browser.close()

await login()
```

</details>

## 6. Scroll infinito y contenido cargado dinámicamente

Muchos sitios cargan más resultados a medida que scrolleás. Un patrón común: scrollear hasta el final, esperar que aparezcan nuevos elementos, y repetir hasta que la cantidad de elementos deje de crecer.

```python
prev = 0
while True:
    await page.mouse.wheel(0, 2000)
    await page.wait_for_timeout(1000)
    actual = await page.locator('.item').count()
    if actual == prev:
        break
    prev = actual
```

### Ejercicio 3 — Adaptar el patrón de scroll infinito

Adaptá el patrón de scroll de la sección anterior para una página cuyos elementos tienen clase `.product-card`, agregando un límite de 10 iteraciones como máximo (para evitar loops infinitos si el sitio nunca deja de cargar).

<details>
<summary>💡 Ver solución</summary>

```python
prev = 0
for _ in range(10):
    await page.mouse.wheel(0, 2000)
    await page.wait_for_timeout(1000)
    actual = await page.locator('.product-card').count()
    if actual == prev:
        break
    prev = actual
```

</details>

## 7. El mismo flujo con Selenium

```bash
pip install selenium
```

Desde Selenium 4, el manejo del driver del navegador es automático (Selenium Manager), sin necesitar descargar el chromedriver a mano.

```python
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = webdriver.ChromeOptions()
options.add_argument('--headless=new')

driver = webdriver.Chrome(options=options)
driver.get('http://quotes.toscrape.com/js/')

WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, 'quote'))
)

quotes = driver.find_elements(By.CLASS_NAME, 'quote')
print(len(quotes))

driver.quit()
```

Documentación oficial: [Selenium con Python](https://selenium-python.readthedocs.io/) · [Esperas explícitas](https://www.selenium.dev/documentation/webdriver/support_features/expected_conditions/)

Notá la diferencia clave: en Selenium las esperas (`WebDriverWait` + `expected_conditions`) son manuales; en Playwright, los locators esperan automáticamente.

### Ejercicio 4 — Extraer autores con Selenium

Usando el driver ya configurado arriba, escribí el código para extraer, de cada `.quote`, el texto del `.author` (usando `find_element` dentro de cada elemento de `quotes`).

<details>
<summary>💡 Ver solución</summary>

```python
autores = [q.find_element(By.CLASS_NAME, 'author').text for q in quotes]
autores
```

</details>

## 8. Modo headless y capturas de pantalla

El modo headless (`headless=True` en Playwright, `--headless=new` en Selenium) corre el navegador sin interfaz gráfica — más rápido y liviano, ideal para servidores y CI. Para depurar, es útil tomar una captura:

```python
await page.screenshot(path='captura.png')  # Playwright
driver.save_screenshot('captura.png')       # Selenium
```

## Mini-proyecto: comparar Playwright y Selenium

1. Elegí una página con contenido dinámico (puede ser `quotes.toscrape.com/js/` u otra de práctica).
2. Extraé el mismo dato (por ejemplo, todas las citas y autores) primero con Playwright y después con Selenium.
3. Medí el tiempo de cada script con `time.time()` y compará.
4. Documentá, en 3-4 líneas, qué herramienta te resultó más simple y por qué.

**Entregable:** notebook o script con ambas implementaciones y la comparación de tiempos.

---

**Seguís en:** *Colab 3 — Rate limiting, ética/legal y proyecto de competitive intelligence*